# Day 52 — Solutions: Scalability with Dask

We build a local CSV fixture, verify a Dask reduction against pandas, measure persisted reuse, and show an optional local-client boundary.

Contents
- Worked reference 1: Verify a local Dask reduction against pandas
- Worked reference 2: Measure persisted reuse honestly
- Worked reference 3: Bound and close an optional local client

---

Worked reference for Exercise 1 — Port pandas to Dask

> Generated from `python/ds-60day/solutions/day52_scalability_dask/day52_solutions.md` by `scripts/build_solution_notebooks.py`. Edit the Markdown source, then regenerate this notebook.


In [ ]:
from pathlib import Path

import dask.dataframe as dd
import numpy as np
import pandas as pd
from dask.diagnostics import ProgressBar

artifact_dir = Path("artifacts/day52")
artifact_dir.mkdir(parents=True, exist_ok=True)
csv_path = artifact_dir / "sales.csv"
rng = np.random.default_rng(52)
source = pd.DataFrame(
    {
        "store": rng.choice(["north", "south", "west"], size=20_000),
        "amount": rng.uniform(1, 100, size=20_000),
        "qty": rng.integers(1, 8, size=20_000),
    }
)
source.to_csv(csv_path, index=False)

ddf = dd.read_csv(str(csv_path), blocksize="64KB")
pipe = (
    ddf[ddf.amount > 0]
    .assign(unit=lambda frame: frame.amount / frame.qty)
    .groupby("store")
    .unit.mean()
)
with ProgressBar():
    result = pipe.compute().sort_index()

expected = (
    source[source.amount > 0]
    .assign(unit=lambda frame: frame.amount / frame.qty)
    .groupby("store")
    .unit.mean()
    .sort_index()
)
assert result.index.tolist() == expected.index.tolist()
np.testing.assert_allclose(
    result.to_numpy(), expected.to_numpy(), rtol=1e-12
)
print(result)


Explanation
- `dd.read_csv` constructs a lazy graph over a concrete local fixture
- `compute` triggers execution; the pandas result is an independent oracle
- A small block size exposes partition behavior without requiring large data

---

Worked reference for Exercise 2 — Persisted reuse timing


In [ ]:
import time

def timed_mean(frame) -> tuple[float, float]:
    started = time.perf_counter()
    value = float(frame.amount.mean().compute())
    return value, time.perf_counter() - started

uncached_value, uncached_seconds = timed_mean(ddf)
persisted = ddf.persist(scheduler="threads")
cached_value, cached_seconds = timed_mean(persisted)
repeat_value, repeat_seconds = timed_mean(persisted)

assert uncached_value == cached_value == repeat_value
print(
    {
        "first_uncached_seconds": round(uncached_seconds, 4),
        "first_persisted_seconds": round(cached_seconds, 4),
        "repeat_persisted_seconds": round(repeat_seconds, 4),
    }
)


Notes
- Include the first persistence cost when comparing one-off work
- Reuse can help only when later actions share the persisted graph
- Release persisted references and close clients when the analysis ends

---

Worked reference for Exercise 3 — Dashboard


In [ ]:
from dask.distributed import Client

# Threads avoid Windows process-spawn boilerplate in a notebook. Enable the
# dashboard intentionally after confirming that the configured port is free.
client = Client(processes=False, dashboard_address=None, n_workers=2)
try:
    cached = client.persist(ddf)
    row_count = int(cached.shape[0].compute())
    assert row_count == len(source)
    print(client)
finally:
    client.close()


Guidance
- This smoke run disables the dashboard. Recreate the client with `dashboard_address=":8787"` only when you intentionally want the UI
- Use the dashboard’s Task Stream to see long‑running tasks
- Profile memory to detect spilling; consider repartitioning: `ddf = ddf.repartition(npartitions=desired)`
- Close client when done to free resources: `client.close()`

---

<!-- BEGIN ADVANCED PYTHON CONCEPT ENRICHMENT -->

## Solution reasoning lens

A strong solution is not merely code that produces one plausible
output. It establishes a chain from input contract to operation to
verification:

1. **lazy expression:** describes tasks and dependencies; inspect partitions/graph before execution.
2. **`.compute()`:** executes the graph and materializes the result in local memory.
3. **chunk state + combine:** keeps sufficient statistics such as sum/count rather than averaging chunk averages incorrectly.
4. **Verification:** Compare the result with an independent invariant, baseline, or failure case before interpreting it.

**Why this approach is appropriate:** Associative reducers establish correctness independent of execution engine; lazy-graph inspection then explains when Dask performs work.

**Useful alternative:** Chunked pandas or PyArrow scanners may be simpler; DuckDB can push filters/aggregations into an embedded analytical engine.

**Trade-off:** More partitions improve available parallelism but increase scheduler overhead; large partitions reduce overhead while increasing peak memory.

**Edge case to test:** Skewed keys, global sorts/shuffles, dtype inference drift, empty chunks, and a final result too large for memory defeat naive scaling.

**Evidence of correctness:** Reconcile results with pandas/standard library, inspect graph and partition sizes, measure repeated runs, bound memory/final output, and prove reducers are associative.

When comparing your attempt with the reference, explain which of these
decisions your code made explicitly. If the reference makes a different
choice, compare the contracts and evidence before deciding that one
version is universally better.

<!-- END ADVANCED PYTHON CONCEPT ENRICHMENT -->

## Exercise-by-exercise reasoning map

This map connects every learner prompt to a reasoning path. Read the
explanation before copying code: the goal is to understand the assumptions,
the evidence that validates the result, and the edge cases that can make an
apparently correct implementation fail.

### Exercise 1 — Original lesson practice

**Prompt:** Read a large local CSV with Dask and compute groupby aggregations.

**How to reason about it:** Generate or use a local CSV, begin small, and compare Dask aggregation with pandas before scaling. Dask expressions are lazy until `compute`, so measure materialization separately.

Use the worked reference earlier in this file, then change one boundary
condition and rerun the stated checks. A copied output is not evidence
unless you can explain why that output follows from the inputs.

**Verify:** Practice 1 — lazy task graphs, partitions, bounded reducers, and scale evidence — generate/read a named local CSV, print Dask partition count and groupby result, and assert keys/values match pandas within 1e-10 after deterministic sorting.

### Exercise 2 — Original lesson practice

**Prompt:** Persist the DataFrame and compare repeated timings with and without persistence.

**How to reason about it:** Persist is beneficial only for reused intermediates and consumes memory. Time a first materialization plus repeated downstream actions, then close distributed clients and release references.

Use the worked reference earlier in this file, then change one boundary
condition and rerun the stated checks. A copied output is not evidence
unless you can explain why that output follows from the inputs.

**Verify:** Practice 2 — lazy task graphs, partitions, bounded reducers, and scale evidence — record at least three uncached and persisted repeated timings for the same aggregation, Dask graph/partition sizes, and result equality; close the local client and avoid claiming a speedup from one run.

### Exercise 3 — Original lesson practice

**Prompt:** Implement the same reduction with `pandas.read_csv(..., chunksize=...)` and compare memory and elapsed time.

**How to reason about it:** Chunked pandas should carry partial sums and counts rather than concatenate all chunks. Verify the final reduction against an in-memory result and handle missing/group-key semantics consistently.

Use the worked reference earlier in this file, then change one boundary
condition and rerun the stated checks. A copied output is not evidence
unless you can explain why that output follows from the inputs.

**Verify:** Practice 3 — lazy task graphs, partitions, bounded reducers, and scale evidence — for identical CSV and aggregation, print pandas-chunked and Dask results, elapsed-time samples, and measured peak memory; assert sorted numeric outputs match within 1e-10.

### Exercise 4 — Task-graph tracing

**Prompt:** Build two aggregations from the same lazy Dask DataFrame, inspect their task graphs, and compare separate computes with one combined `dask.compute` call.

**Reasoning before implementation:** Building an expression does not read all data. Combining terminal computations can share upstream work without persisting the entire frame.

Record graph/task counts and wall time, but treat timings as local evidence
rather than universal benchmarks. `dask.compute(result_a, result_b)` can
optimize shared dependencies in one execution. Two separate `.compute()` calls
may reread and redo work unless data or intermediates are cached/persisted.

Keep output bounded; computing a small aggregation is different from calling
`compute()` on the full large DataFrame.

**Why this matters:** The result should survive a fresh-kernel rerun and
a deliberately chosen boundary case. If it does not, revisit the
assumption or data boundary rather than hiding the failure.

**Verify:** Task-graph tracing — print task counts/graph keys for two separate aggregations and one combined dask.compute call; assert all numeric results match and report shared-prefix work without promising scheduler-specific task elimination.

### Exercise 5 — Partition-skew diagnosis

**Prompt:** Create a group key where one value owns most rows. Measure partition sizes and groupby runtime, then propose repartitioning or algorithm changes.

**Reasoning before implementation:** A balanced row count before a shuffle does not guarantee balanced work after grouping; one hot key can become a straggler.

Report the largest group and partition relative to the median. Salting a hot
key can distribute an associative partial aggregation, followed by a second
combine, but it changes implementation complexity. More partitions alone do
not split one indivisible group operation automatically.

If the output is tiny, pre-aggregate within input partitions before the
shuffle. Preserve exactness and test the salted/two-stage result against pandas.

**Why this matters:** The result should survive a fresh-kernel rerun and
a deliberately chosen boundary case. If it does not, revisit the
assumption or data boundary rather than hiding the failure.

**Verify:** Partition-skew diagnosis — print rows/bytes per partition, max-to-median skew ratio, dominant-key support, and repeated groupby timings before/after the chosen mitigation; assert outputs remain equal.

### Exercise 6 — Reducer correctness

**Prompt:** Implement a mergeable mean/variance state for chunks and prove it matches NumPy across different chunk boundaries, including an empty chunk.

**Reasoning before implementation:** A mean alone is not mergeable without support. Carry count, mean, and M2 (sum of squared deviations) using a stable combine formula.

The state must define an identity for an empty chunk and an associative combine
operation (within floating-point tolerance). Welford/Chan formulas combine
`count`, `mean`, and `M2` without concatenating values.

Test one chunk, one-row chunks, uneven chunks, large-offset values, and an
empty input policy. This same reasoning underlies correct distributed custom
aggregations; averaging partition means is wrong when partition sizes differ.

**Why this matters:** The result should survive a fresh-kernel rerun and
a deliberately chosen boundary case. If it does not, revisit the
assumption or data boundary rather than hiding the failure.

**Verify:** Reducer correctness — test the mergeable state on empty, singleton, unequal, and reordered chunks; print count/mean/variance and assert agreement with NumPy within 1e-12 for every partitioning.
